# C3-ROI + Spatial Attention V1
Chỉ sửa `FOLD`; các fold còn lại dùng nguyên cấu hình đã khóa.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
import zipfile

FOLD = 1  # DOI DUY NHAT DONG NAY: 1, 2, 3, 4 hoac 5
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/data')
WORKSPACE = Path('/content/c3_attention_workspace')
DATA_ZIP = DRIVE_DATA_ROOT / 'C3_ROI_V1_READY_TO_TRAIN_512_JPEG_V2.zip'
CODE_ZIP = DRIVE_DATA_ROOT / 'C3_ROI_ATTENTION_COLAB_CODE_FINAL_V3.zip'

assert FOLD in range(1, 6)
assert DATA_ZIP.is_file(), DATA_ZIP
assert CODE_ZIP.is_file(), CODE_ZIP
WORKSPACE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DATA_ZIP) as zf:
    zf.extractall(WORKSPACE)
with zipfile.ZipFile(CODE_ZIP) as zf:
    zf.extractall(WORKSPACE)

# Data ZIP co the co them mot thu muc C3_ROI_V1; dua code va data ve cung root.
roi_candidates = sorted(WORKSPACE.glob('**/c3_roi/cache/C3_ROI_V1/roi'))
assert roi_candidates, 'Khong tim thay ROI cache sau khi giai nen data ZIP'
PROJECT_ROOT = roi_candidates[0].parents[3]
for component in ('p1_baseline', 'c3_roi', 'c3_roi_attention'):
    source = WORKSPACE / component
    destination = PROJECT_ROOT / component
    if source.exists() and source.resolve() != destination.resolve():
        import shutil
        shutil.copytree(source, destination, dirs_exist_ok=True)
os.chdir(PROJECT_ROOT)
print('READY', PROJECT_ROOT, 'FOLD', FOLD)

In [ ]:
%cd /content/c3_attention_workspace
!pip -q install -r p1_baseline/requirements.txt

In [ ]:
from pathlib import Path
roi_count = sum(1 for p in Path('c3_roi/cache/C3_ROI_V1/roi').rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
print('ROI images:', roi_count)
assert roi_count == 14036, f'Can du 14036 ROI images, hien co {roi_count}'

In [ ]:
!python -m c3_roi_attention.colab_runner --fold {FOLD} --drive-data-root "{DRIVE_DATA_ROOT}"

In [ ]:
import json
run_dir = DRIVE_DATA_ROOT / 'c3_attention_runs' / f'C3_ROI_ATTN_V1_FOLD_{FOLD}'
print('Saved at:', run_dir)
state = run_dir / 'run_state.json'
print(json.loads(state.read_text()) if state.is_file() else 'run_state.json chua duoc ghi')